# Задача 1: Лингвистика и синтаксический разбор

Начальный пример для работы с задачей.

In [6]:
# Установка необходимых библиотек
!pip install natasha
!pip install -U pymorphy2==0.8
!pip install matplotlib
!pip install pandas

   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.1 MB 1.8 MB/s eta 0:00:04
   ---------- ----------------------------- 1.8/7.1 MB 3.1 MB/s eta 0:00:02
   ------------------------- -------------- 4.5/7.1 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 7.1/7.1 MB 8.3 MB/s  0:00:01

  Attempting uninstall: pymorphy2

    Found existing installation: pymorphy2 0.9.1

   -------------------- ------------------- 1/2 [pymorphy2]
    Uninstalling pymorphy2-0.9.1:
   -------------------- ------------------- 1/2 [pymorphy2]
      Successfully uninstalled pymorphy2-0.9.1
   -------------------- ------------------- 1/2 [pymorphy2]
   -------------------- ------------------- 1/2 [pymorphy2]
   ---------------------------------------- 2/2 [pymorphy2]



## Импорт необходимых библиотек

In [7]:
# Стандартные библиотеки
from collections import Counter
import json
import re

# Библиотеки для визуализации
import matplotlib.pyplot as plt
import pandas as pd
# Лингвистические библиотеки
from natasha import (
    Segmenter,
    MorphVocab,
    NewsMorphTagger,
    NewsSyntaxParser,
    Doc
)
from pymorphy3 import MorphAnalyzer


## Загрузка данных

In [8]:
def load_texts(file_path):
    """
    Загружает текстовые данные из файла.
    
    Args:
        file_path: путь к файлу с текстами
    
    Returns:
        list: список предложений
    """
    # TODO: реализовать загрузку текстов
    # Подсказка: можно использовать json.load() если данные в JSON формате
    # или просто читать построчно если это текстовый файл
    # Пример для текстового файла:
    # with open(file_path, 'r', encoding='utf-8') as f:
    #     text = f.read()
    #     sentences = re.split(r'[.!?]+', text)
    #     return [s.strip() for s in sentences if s.strip()]
    
    with open(file_path, 'r', encoding='utf-8') as f:
         text = f.read()
         sentences = re.split(r'[.!?]+', text)
         return [s.strip() for s in sentences if s.strip()]

In [9]:
# Пример использования
# texts = load_texts('data/texts.txt')
# print(f"Загружено предложений: {len(texts)}")

texts = load_texts('news_dataframe.txt')
print(f"Загружено предложений: {len(texts)}")

Загружено предложений: 93


## Инициализация инструментов

In [10]:
# Инициализация инструментов natasha

segmenter = Segmenter()
morph_vocab = MorphVocab()
morph_tagger = NewsMorphTagger(morph_vocab)
syntax_parser = NewsSyntaxParser(morph_vocab)

# Инициализация pymorphy3
morph = MorphAnalyzer()


ModuleNotFoundError: No module named 'pkg_resources'

## Синтаксический разбор предложений

In [35]:
def parse_sentence(sentence, segmenter, morph_tagger, syntax_parser):
    """
    Выполняет синтаксический разбор предложения и выделяет подлежащее и сказуемое.
    
    Args:
        sentence: строка с предложением
        segmenter: объект Segmenter из natasha
        morph_tagger: объект MorphTagger из natasha
        syntax_parser: объект SyntaxParser из natasha
    
    Returns:
        tuple: (подлежащее, сказуемое) или (None, None) если не найдено
    """
    # TODO: реализовать синтаксический разбор с помощью natasha
    # Подсказка:
    # 1. Создать объект Doc: doc = Doc(sentence)
    # 2. Применить segmenter: doc.segment(segmenter)
    # 3. Применить morph_tagger: doc.tag_morph(morph_tagger)
    # 4. Применить syntax_parser: doc.parse_syntax(syntax_parser)
    # 5. Найти подлежащее (токен с зависимостью 'nsubj' или 'nsubj:pass')
    #    Пример: для токена token, проверить token.rel == 'nsubj'
    # 6. Найти сказуемое (токен с зависимостью 'root' или связанный с подлежащим)
    #    Пример: для токена token, проверить token.rel == 'root'
    # Доступ к токенам: doc.sents[0].tokens
    # Доступ к тексту токена: token.text
    # Доступ к синтаксическим зависимостям: token.rel для отношения, token.head для головы
    ...
    doc = Doc(sentence)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.parse_syntax(syntax_parser)
    if not doc.sents:
        return None, None
    
    # Берем первое предложение
    first_sentence = doc.sents[0]
    
    # Инициализируем переменные для подлежащего и сказуемого
    subject = None
    predicate = None
    
    # Проходим по всем токенам (словам) в предложении
    for token in first_sentence.tokens:
        # Ищем подлежащее (токен с зависимостью 'nsubj' или 'nsubj:pass')
        # nsubj = nominal subject (именное подлежащее)
        # nsubj:pass = nominal subject passive (подлежащее при пассиве)
        if token.rel in ['nsubj', 'nsubj:pass']:
            subject = token.text
        
        # Ищем сказуемое (токен с зависимостью 'root')
        # root - корневое слово предложения (обычно глагол-сказуемое)
        if token.rel == 'root':
            predicate = token.text
    
    # Если подлежащее не найдено, пробуем альтернативный способ:
    # ищем слово, от которого зависит сказуемое
    if subject is None and predicate is not None:
        for token in first_sentence.tokens:
            # Если у токена голова (главное слово) - это сказуемое
            if token.head is not None and token.head.text == predicate:
                # И проверяем, что это подлежащее
                if token.rel in ['nsubj', 'nsubj:pass']:
                    subject = token.text
                    break
    
    return subject, predicate

In [6]:
def build_cooccurrence_dependencies(texts, segmenter, morph_tagger, syntax_parser):
    """
    Проверяет, является ли слово сказуемым.
    
    Args:
        word: слово для проверки
        morph: объект MorphAnalyzer из pymorphy3
    
    Returns:
        bool: True если слово является сказуемым
    """
    # TODO: реализовать проверку с помощью pymorphy3
    # Подсказка:
    # 1. Использовать morph.parse(word) для получения морфологического разбора
    # 2. Проверить, что часть речи - сказуемое
    # 3. Проверить, что слово может быть сказуемым (например, в краткой форме)
    
    parsed = morph.parse(word)
    for p in parsed:
        # 2. Проверить, что часть речи - сказуемое
        # В pymorphy3 часть речи хранится в tag.POS
        # Сказуемое в русском языке обычно выражается глаголом (VERB)
        # Также инфинитив (INFN) и краткие причастия (PRTS) могут быть сказуемыми
        
        pos = p.tag.POS
        
        # Проверяем, является ли часть речи глаголом (личная форма)
        if pos == 'VERB':
            return True
        
        # Проверяем инфинитив
        if pos == 'INFN':
            return True
        
        # Проверяем краткие причастия (например: "сделано", "решена")
        if pos == 'PRTS' and 'short' in p.tag:
            return True
        
        # 3. Проверить, что слово может быть сказуемым (например, в краткой форме)
        # Краткие прилагательные тоже могут быть сказуемыми (например: "он умён")
        if pos == 'ADJF' and 'short' in p.tag:
            return True
    
    return False

In [34]:
def build_cooccurrence_dependencies(texts, segmenter, morph_tagger, syntax_parser):
    """
    Строит зависимости совместных употреблений подлежащих и сказуемых.
    
    Args:
        texts: список предложений
        segmenter, morph_tagger, syntax_parser: объекты из natasha
        morph: объект MorphAnalyzer из pymorphy3
    
    Returns:
        Counter: счетчик пар (подлежащее, сказуемое)
    """
    # TODO: реализовать построение зависимостей
    # Подсказка:
    # 1. Для каждого предложения вызвать parse_sentence
    # 2. Если найдены подлежащее и сказуемое, добавить пару (подлежащее, сказуемое) в список
    # 3. Использовать collections.Counter для подсчета частот
    # Пример: counter = Counter([('студент', 'учится'), ('преподаватель', 'объясняет'), ...])
    
    cooccurrences = []
    
    # Ваш код здесь
    # for sentence in texts:
    #     subject, predicate = parse_sentence(sentence, segmenter, morph_tagger, syntax_parser)
    #     if subject and predicate:
    #         cooccurrences.append((subject, predicate))

    for sentence in texts:
        subject, predicate = parse_sentence(sentence, segmenter, morph_tagger, syntax_parser)
        
        # 2. Если найдены подлежащее и сказуемое, добавить пару в список
        if subject and predicate:
            cooccurrences.append((subject, predicate))
            
    return Counter(cooccurrences)

## Визуализация результатов

In [33]:
def visualize_results(counter, top_n=20):
    """
    Визуализирует результаты анализа.
    
    Args:
        counter: Counter с парами (подлежащее, сказуемое)
        top_n: количество топовых сочетаний для отображения
    """
    # TODO: реализовать визуализацию
    # Подсказка:
    # 1. Получить top_n наиболее частых сочетаний
    # 2. Построить график (столбчатая диаграмма, например)
    # 3. Можно использовать matplotlib / seaborn / pandas / любое другое решенеие для визуализации
    ...
    top_pairs = counter.most_common(top_n)
    
    if not top_pairs:
        print("Нет данных для визуализации")
        return
    
    # Подготавливаем данные
    labels = [f"{subj} → {pred}" for (subj, pred), _ in top_pairs]
    values = [count for _, count in top_pairs]
    
    # 2. Построить график
    plt.figure(figsize=(10, 6))
    
    # Вертикальная столбчатая диаграмма
    plt.bar(range(len(labels)), values, color='steelblue', edgecolor='black')
    
    # Настройка подписей
    plt.xlabel('Пары (подлежащее → сказуемое)', fontsize=12)
    plt.ylabel('Частота', fontsize=12)
    plt.title(f'Топ-{top_n} частотных пар подлежащее-сказуемое', fontsize=14)
    
    # Поворачиваем подписи, чтобы они не налезали друг на друга
    plt.xticks(range(len(labels)), labels, rotation=45, ha='right', fontsize=8)
    
    # Добавляем значения на столбцы
    for i, v in enumerate(values):
        plt.text(i, v + 0.1, str(v), ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## Основной код выполнения задачи

In [11]:
# Загрузка данных
texts = load_texts('news_dataframe.txt')
print(f"Загружено предложений: {len(texts)}")

# Построение зависимостей
cooccurrences = build_cooccurrence_dependencies(
     texts, 
     segmenter, 
     morph_tagger, 
     syntax_parser, 
     morph
)

# Вывод результатов
print("\nТоп-10 наиболее частых сочетаний:")
for (subject, predicate), count in cooccurrences.most_common(10):
     print(f"{subject} - {predicate}: {count}")

# Визуализация
visualize_results(cooccurrences, top_n=20)


Загружено предложений: 93


NameError: name 'build_cooccurrence_dependencies' is not defined